# Inlämningsuppgift 1 del 3 My Tistelberg

Min idé för detta projekt var en app för servicetekniker ute i fältet eftersom jag har erfarenhet utav att vara en servicetekniker ute i fältet så vet jag att det hade varit uppskattat att ha en sådan app för att slippa slå i manualen hela tiden utan att få hjälp med sin fundering för att snabbt kunna jobba vidare. 
Men nu är det så att servicemanualer är inte publika utan det jag hittade var användarmanualer så då tänkte jag att jag istället skulle göra en app för användaren då jag tror det finns en sot okunskap vid maskiner och ingen läser ju manualen. 
Eftersom vi nyligen köpte en tvättmaskin valde jag att köra på tvättmaskinsmanualer. Jag tror detta hade haft ett värde för ett tvättmaskinsföretag att ha för att göra kunden mer säker på maskinen men också skulle kunna minimera samtal till kuntjänsten för mindre komplexa frågor men också kunna hänvisa till kundtjänsten om det skulle vara något fel en servicetekniker skulle behöva åka ut på och på så sätt få mer försäljning på service. 

Jag började med 5st manualer men eftersom min quota tog slut väldigt snabbt så bregränsade jag antalet och har nu bara en manual. Men jag har gjort så att det går att skala om man skulle vilja. 

## Importer

In [1]:
from pypdf import PdfReader
from pathlib import Path
import re
from google import genai
import os
from google.genai import types
from dotenv import load_dotenv
import json
import time
import numpy as np
import pdfplumber
import fitz
from pathlib import Path


Sparade min API nyckel i ett separtat dokument för att inte publicera den.

In [2]:
load_dotenv()

True

In [3]:
client = genai.Client(api_key=os.getenv("API_KEY"))

# RAG-delen

## Inläsning av dokumenten

Jag fick testa flera olika sorters pdf-inläsningar då jag märkte att den första jag använde (den från boken) inte fungerade för mig eftersom jag läser in en manual och inte bara en vanlig text. 
Efter flera olika försök hittade jag att fitz fungerar bra på manualer och den har parametrar man kan ställa in hur den ska läsa in paragrafer (vänster till höger). 

In [4]:
# Läser in alla pdferna som ligger i mappen "Dokument till RAG" i mitt repository
# folder_path = Path("Dokumenten till RAG")
#pdf_files = list(folder_path.glob("*.pdf"))
#documents = []

#for pdf in folder_path.glob("*.pdf"): #läser in alla alla dokument som slutar med pdf
#    reader = PdfReader(pdf)

#    text = ""
#    for page in reader.pages[3:]: # Väljer att inte läsa in framsidan och orelevanta sidor i början
#        page_text = page.extract_text()
        # Lägger bara till om det verkligen finns någon text på sidan
#        if page_text:
#            text += page_text + "\n"
    
    # sparar från vilket dokument texten kommer ifrån
#    documents.append({
#        "source": pdf.name,
#        "text": text
#    })


In [5]:
#import pdfplumber

#documents = []

#for pdf in folder_path.glob("*.pdf"):
#    text = ""

#    with pdfplumber.open(pdf) as pdf_file:
#        for page in pdf_file.pages[3:]:
#            page_text = page.extract_text(x_tolerance=2, y_tolerance=2)

#            if page_text:
#                text += page_text + "\n\n"   # viktigt: dubbla radbrytningar

#    documents.append({
#        "source": pdf.name,
#        "text": text
#    })

In [6]:
folder_path = Path("Dokumenten till RAG")
documents = []

for pdf_path in folder_path.glob("*.pdf"):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc[3:]:
        blocks = page.get_text("blocks")
        
        # sortera block: uppifrån och sedan vänster till höger
        blocks = sorted(blocks, key=lambda b: (b[0], b[1]))

        page_parts = []
        for b in blocks:
            block_text = b[4].strip()
            if not block_text:
                continue
        page_parts = []
        for b in blocks:
            block_text = b[4].strip()
            if block_text:
                page_parts.append(block_text)
    
        text += "\n\n".join(page_parts) + "\n\n"

    documents.append({
        "source": pdf_path.name,
        "text": text
    })

In [7]:
# Kontroll av den texten
print(documents[0]["text"][:2000])

Avsedd användning
Tvättmaskinen är avsedd att användas
• uteslutande för hushållsbruk
• för tvätt av kläder som tål vattentvätt i
maskin
• för drift med kallt kranvatten och vanliga
tvätt- och sköljmedel som är avsedda för
maskintvätt
•Tvättmaskinen kan användas av barn från
8 år, av personer med nedsatt fysisk,
sensorisk eller mental kapacitet samt av
personer som själva saknar tillräcklig
erfarenhet och kunskap, om de övervakas
eller instrueras av en ansvarig person.
Säkerhetsanvisningar
Lämna inte barn utan uppsikt vid
tvättmaskinen.
Håll husdjur borta från tvättmaskinen.
Var alltid torr om händerna när du tar tag i
nätkontakten för att sätta in eller dra ut den
ur uttaget.

Tvättmaskinen är sparsam med förbrukning
av vatten, energi och tvättmedel. På så sätt
skonar du både miljön och hushållskassan. 
Tvätta sparsamt och miljövänligt:
• Överskrid inte rekommenderad
tvättmängd.
Vit- och kulörtvätt ..............................5,5 kg
Syntet ...........................................

Vid steget kontroll av embeddingsen så upptäckte jag att textkvaliten var dålig då det var radbrytningar mitt i ord i pdfen så därför lägger jag till detta steget som ska städa texten för att underlätta embeddningsen i senare skede. 
Sedan under alla iterationer utav detta så har jag hittat mer och mer saker jag vill städa bort eller ändra för att få texten så enkel som möjligt för modellerna i senare skede så denna funktion har växt en del.

In [8]:
def clean_text(text):
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        if not line:
            cleaned_lines.append("")
            continue

        # ta bort rena sidnummer
        if re.fullmatch(r"\d+", line):
            continue

        # ta bort "Ritning 1", "Ritning 7.2" osv
        if re.fullmatch(r"Ritning\s+\d+(\.\d+)?", line, flags=re.IGNORECASE):
            continue

        # ta bort rader som nästan bara är symboler
        if re.fullmatch(r"[\.\-\s•➀➁➂➃➄➅:;/,()\[\]]+", line):
            continue

        # ta bort PDF-brus
        line = re.sub(r"/L\d+[A-Z0-9]*", "", line)

        # normalisera bindestreck
        line = line.replace("–", "-").replace("—", "-")

        # normalisera whitespace inom raden
        line = re.sub(r"\s+", " ", line).strip()

        line = re.sub(r"[\x00-\x1F\x7F]", "", line)
        line = re.sub(r"\s*\.{2,}\s*", " : ", line)

        if line:
            cleaned_lines.append(line)

    # nu jobbar vi på hela texten
    text = "\n".join(cleaned_lines)
#    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # städa upp för många tomrader
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [46]:
# Städar dokumentet
for doc in documents:
    doc["text"] = clean_text(doc["text"])

In [10]:
# Kontroll av den texten efter städning
print(documents[0]["text"][:2000])

Avsedd användning
Tvättmaskinen är avsedd att användas
• uteslutande för hushållsbruk
• för tvätt av kläder som tål vattentvätt i
maskin
• för drift med kallt kranvatten och vanliga
tvätt- och sköljmedel som är avsedda för
maskintvätt
•Tvättmaskinen kan användas av barn från
8 år, av personer med nedsatt fysisk,
sensorisk eller mental kapacitet samt av
personer som själva saknar tillräcklig
erfarenhet och kunskap, om de övervakas
eller instrueras av en ansvarig person.
Säkerhetsanvisningar
Lämna inte barn utan uppsikt vid
tvättmaskinen.
Håll husdjur borta från tvättmaskinen.
Var alltid torr om händerna när du tar tag i
nätkontakten för att sätta in eller dra ut den
ur uttaget.

Tvättmaskinen är sparsam med förbrukning
av vatten, energi och tvättmedel. På så sätt
skonar du både miljön och hushållskassan.
Tvätta sparsamt och miljövänligt:
• Överskrid inte rekommenderad
tvättmängd.
Vit- och kulörtvätt : 5,5 kg
Syntet : 2,5 kg
Fintvätt/siden : 1,0 kg
Ull : 1,5 kg
Vid mindre tvättmängder re

## Första chunkningen

Jag började först men en vanlig meningsgrundad chunking då jag valde att följa exemplet i boken. Men jag insåg att detta inte kommer fungera för mig för manualer har en tendens att hoppa över punkter och är uppdelade på andra sätt som inte fungerade för detta.

Därför valde jag en blockbaserad chunking för att behålla herakin och strukturen på texten. Så först delar jag in texten i olika blocks baserat på 

In [11]:
#chunks = []
#n = 7
#overlap = 3

#for doc in documents: 
#    sentences = re.split(r'(?<=[.!?]) +', doc["text"])

#    for i in range(0, len(sentences), n - overlap):
#        chunk_sentences = sentences[i:i+n]

#        chunk_text = " ".join(chunk_sentences)

#        chunks.append({
#            "text": chunk_text,
#            "source": doc["source"]
#        })

Jag fortsätter att städa lite till då det är lite mer konstiga saker i texten.

In [12]:
def is_noise_line(line: str) -> bool:
    """Filtrerar bort skräp från texten"""
    line = line.strip()

    if not line:
        return False

    # korta konstiga labels från PDF/figurer
    if len(line.split()) <= 2 and len(line) <= 12:
        if re.fullmatch(r"[A-Za-zÅÄÖåäö0-9➀➁➂➃➄➅\-/]+", line):
            return True

    # typ JJJJAAAA
    if re.search(r"(.)\1{3,}", line):
        return True

    return False

Här gör jag en funktion för att ta ut rubrikerna som baserar sig på att det inte ska vara en tom rad, flera rader, börja med liten bokstac, för lång rad eller ser ut som en vanlig mening som slutar med . ! eller ?
Rubrikerna kommer att användas för att dela dokumentet i sektioner.

In [13]:
def is_heading(text: str) -> bool:
    """Klassar meningen som rubrik eller inte"""
    text = text.strip()

    if not text or "\n" in text:
        return False

    if text[0].islower():
        return False

    words = text.split()

    if len(words) > 10 or len(text) > 80:
        return False

    if text.endswith((".", "!", "?")):
        return False

    if re.fullmatch(r"\d+(\.\d+)*\s+[A-ZÅÄÖa-zåäö].*", text):
        return True

    if text.isupper() and 2 <= len(words) <= 8:
        return True

    alpha_ratio = sum(c.isalpha() for c in text) / max(len(text), 1)

    if 2 <= len(words) <= 6 and alpha_ratio > 0.6 and not re.search(r"[;|=]{2,}", text):
        capitalized = sum(w[:1].isupper() for w in words if w)
        if capitalized >= max(1, len(words) - 1):
            return True

    return False

Nästa är en funktion för att dela upp raderna i block för att senare kunna underlätta chunkingen.
Här så ser vi till att behålla textens struktur så paragrafer, listor behålls intakta och inte delas upp. 
Men spec rader så som innehåll i tabeller eller specifikationer kan vara i ett eget block. 
Detta för att i kommande funktion kommer vi att dela in dessa block i sektioner baserat på rubrikerna.

In [14]:
def split_into_blocks(text: str) -> list[str]:
    """Bygger block av raderna. 
    Ett block är en semantisk enhet tex.:
    - Paragraf
    - Lista
    - Spec-rad
    - Rubrik """
    raw_lines = [line.strip() for line in text.splitlines()]
    lines = [line for line in raw_lines if line]

    blocks = []
    current_block = []

    for line in lines:
        if is_noise_line(line):
            continue

        # rubriker egna block
        if is_heading(line):
            if current_block:
                block = "\n".join(current_block).strip()
                if block:
                    blocks.append(block)
                current_block = []
            blocks.append(line)
            continue

        # listpunkt -> behåll i samma block
        if re.match(r"^[-•*]\s+", line):
            current_block.append(line)
            continue

        # tabell/spec-rad -> eget block
        if re.match(r"^[^:]{1,40}:\s+\S+", line):
            if current_block:
                block = "\n".join(current_block).strip()
                if block:
                    blocks.append(block)
                current_block = []
            blocks.append(line)
            continue

        current_block.append(line)

    if current_block:
        block = "\n".join(current_block).strip()
        if block:
            blocks.append(block)

    # Filtrerar bort tomma blocks
    cleaned_blocks = [b for b in blocks if b.strip()]

    return cleaned_blocks

Nästa funktion delar nu upp dokumentet i chunks baserat på blocken.
Det den gör är att den börjar med att dela upp texten i blocks med funktionen ovan sen så delar den upp i chunks i sektioner beroende på rubrikerna. 
Så att allt under samma rubrik ska senare hamna i samma chunk. 
Sedan tar den och itirerar över och lägger till ett antal sektioner i en chunk beroende på ordantalet i sektionen vilket bestäms av target_words och max_word, och så är det överlapp med block (overlap_blocks).

I funktionen skapar vi två typer av texter i slutet, en som ska användas till LLMn när den genererar svar till oss och den texten behåller sitt naturliga språk.
Sedan görs en text som ska användas för embeddingen som struktureras så att den definerar vilket dokument det kommer ifrån, vad som är rubrik och vad som är själva textinnehållet. Varför jag har med från vilket dokument var för att från början använde jag flera dokument men det blev för dyrt med anropen så jag har begränsat till ett dokument nu, men jag bygger så att jag skulle kunna lägga till flera dokument om jag skulle velat.

I slutet av denna funktionen så lägger jag ihop all info i en lista med dict. med all information så att allt är kopplat och texten för embeddingen kan användas och sedan tar man fram vanliga texten och matar LLMn med det så att den kan generera ett svar.

In [ ]:
def chunk_document(
    text: str,
    source: str,
    target_words: int = 220,
    max_words: int = 320,
    overlap_blocks: int = 1
):
    """Denna funktion skapar chunks för dokumentet.
    Det skapas en lista med dict med:
    - text, texten till LLM-modellen
    - embedding_text, texten till embedding modellen
    - source, för möjlig felsökning
    - heading, för möjlig felsökning
    - section_id, för möjlig felsökning
    - word_count, för möjlig felsökning"""
    
#-------------- Sektionsuppbyggnad -------------------------------------------------------
    blocks = split_into_blocks(text)

    sections = []
    current_heading = None
    current_blocks = []

    for block in blocks:
        if is_heading(block):
            if current_blocks:
                sections.append({
                    "heading": current_heading,
                    "blocks": current_blocks
                })
            current_heading = block
            current_blocks = []
        else:
            current_blocks.append(block)

    # Sista sektionen måste sparas manuellet
    if current_blocks:
        sections.append({
            "heading": current_heading,
            "blocks": current_blocks
        })

    chunks = []

#----------------------- Chunkingen --------------------------------------------------------
    for sec_idx, section in enumerate(sections):
        heading = section["heading"]
        sec_blocks = section["blocks"]

        i = 0
        while i < len(sec_blocks):
            chunk_blocks = []
            word_count = 0
            j = i

            while j < len(sec_blocks):
                block = sec_blocks[j]
                block_words = len(block.split()) 

                # om vi redan har nått target, bryt hellre än att göra chunken för stor
                if chunk_blocks and word_count >= target_words:
                    break

                # Bryta om orden överstiger maxgränsen
                if chunk_blocks and word_count + block_words > max_words:
                    break

                chunk_blocks.append(block)
                word_count += block_words
                j += 1

            # Skapar en text som ska användas till LLMn som är mer naturligt språk än den som skickas till embeddingen
            chunk_body = "\n\n".join(chunk_blocks).strip()

            if heading:
                chunk_text = f"{heading}\n\n{chunk_body}"
            else:
                chunk_text = chunk_body

            # Gör texten som ska embeddas som ska ha all information vilket då blir tydligt semantiskt för optimal embedding
            embedding_text = (
                f"Dokument: {source}\n"
                f"Rubrik: {heading or 'Ingen rubrik'}\n"
                f"Innehåll:\n{chunk_body}"
            )

            # Här kopplar vi ihop chunk_text och embed_text för att hämtningen ska kunna använda embedding_text 
            # men chatbotten ska få chunk_texten att generera svar ifrån
            # Resten av sakerna har används/kan användas vid debugging
            chunks.append({
                "text": chunk_text,
                "embedding_text": embedding_text,
                "source": source,
                "heading": heading,
                "section_id": sec_idx,
                "word_count": len(chunk_text.split())
            })

            if j == i:
                i += 1
            else:
                i = max(j - overlap_blocks, i + 1) # Gör ett overlap i chunksen för om någon info skulle vara vid gränsen ex problemet i chunk 1 och svaret i chunk 2

    return chunks

Här under utförs själva chunkingen med ovan funktioner. Också här varför det står documents är att jag från början hade tänkt ha flera och som det ser ut nu skulle jag kunna lägga till fler vid behov.

In [16]:
all_chunks = []

for doc in documents:
    doc_chunks = chunk_document(doc["text"], doc["source"])
    all_chunks.extend(doc_chunks)

In [17]:
# Kontroll av hur många chunks som skapades
print(len(all_chunks))

81


## Embeddings

In [18]:
def create_embeddings(text,
                      model="gemini-embedding-001",
                      task_type="SEMANTIC_SIMILARITY"):
    """Skickar texten till embeddingmodellen som retunerar en vektor"""
    return client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )

Eftersom jag fick problem med att jag överskred min quota så har jag minskat antal manualer från 5 till 1 samt satt in att spara embeddnings mellan varje lyckad embedding för att inte behöva köra om från början om jag uppnår min quota. Jag satt även in en liten fördröjning mellan chunksen för att minska belastningen.

In [19]:
save_path = "embeddings.json"

# ladda in redan sparade embeddings om filen finns
if os.path.exists(save_path):
    with open(save_path, "r", encoding="utf-8") as f:
        embeddings = json.load(f)    
else:
    embeddings = []

already_done = {item["embedding_text"] for item in embeddings}

for chunk in all_chunks:
    if chunk["text"] in already_done:
        continue

    try:
        response = create_embeddings(chunk["embedding_text"])
    
        item = {
        "text": chunk["text"],
        "embedding_text": chunk["embedding_text"],
        "source": chunk["source"],
        "embedding": response.embeddings[0].values
        }

        embeddings.append(item)

        # spara direkt efter varje lyckad embedding
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(embeddings, f, ensure_ascii=False, indent=2)

        print(f"Sparade chunk från {chunk['source']}")

        time.sleep(1) # liten paus för att minska belastningen

    except Exception as e:
        print("Stopp:", e)
        break

    # Gör om embeddings till numpy‑matris (för beräkning) och metadata‑lista
    chunk_vectors = np.array([item["embedding"] for item in embeddings])
    records = [{"text": item["text"], "source": item["source"]} for item in embeddings]

Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 9000388437_D.pdf
Sparade chunk från 900038843

## Semantisk sökning funktionerna

In [20]:
def cosine_similarity(vector, matrix):
    """Jämför frågan med chunksen på deras embedding """
    # Normalisera vektorer
    vector_norm = vector / np.linalg.norm(vector)
    matrix_norm = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    # Beräkna similarity för alla på en gång
    return np.dot(matrix_norm, vector_norm.T).flatten()

In [21]:
def semantic_search(query, top_k=5):
    """Tar frågan och retunerar de top_k mest semantiska chunksen

        Return:
            List[dict]: [
                {
                    "score": float,
                    "text": str,
                    "source": str
                }
        ]"""
    query_response = create_embeddings(query)
    query_vector = np.array(query_response.embeddings[0].values).reshape(1,-1)

    scores = cosine_similarity(query_vector, chunk_vectors)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "score": float(scores[idx]),
            "text": records[idx]["text"],
            "source": records[idx]["source"]
        })

    return results

## Kontroll av embeddingsen

Först började jag med att kontrollera vad jag tyckte om embedding modellen, lite hur den fungerade och om den var bra. Detta började jag med genom att bara kontrollera meningar mot varandra. Vet inte hur mycket det gav egentligen men var lite kul att jämföra meningar och de blev förvånadsvärt nära i poäng men som jag förstod det så är det pga att det är samma språk och typ samma struktur på meningarna. 

Efter ha testat detta så kommenterade jag bort testen för att slippa köra det varje gång jag körde om koden.

In [22]:
def similarity(text_a, text_b):
    """Jämför två texter hur semantiskt nära de är varandra"""
    emb_a = np.array(create_embeddings(text_a).embeddings[0].values)
    emb_b = np.array(create_embeddings(text_b).embeddings[0].values)

    return np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))

In [23]:
pairs = [
    ("Maskinen fylls inte med vatten.", "Apparaten fylls inte med vatten korrekt."),
    ("Rengör maskinen regelbundet.", "Vänta tills nätspänningen är stabil."),
    ("Avbryt ett pågående program.", "Stoppa maskinen medan den kör."),
]

#for a, b in pairs:
#    print(f"{a}\n{b}\nLikhet: {similarity(a,b):.2f}\n")


In [24]:
pairs = [
    ("Rengör maskinen regelbundet.", "Maskinen har en maximal kapacitet på 8 kg."),
    ("Avbryt ett pågående program.", "Barn ska inte leka med apparaten."),
    ("Maskinen fylls inte med vatten.", "Apparaten fylls inte med vatten korrekt."),
    ("Hunden är ett djur med fyra ben, en svans och den gillar att äta ben.", "Rengör maskinen regelbundet."),
    ("Maskinen fylls inte med vatten.", "Använd flytande tvättmedel för ullprogram.")
]

#for a, b in pairs:
#    print(f"{a}\n{b}\nLikhet: {similarity(a,b):.2f}\n")

Här skapar jag en funktion för själva retrievalen av chunksen

In [25]:
def retrieval_evaluation(query):
    """Denna funktion tar fram top_k chunks och skriver ut deras info:
    - Semantik score
    - Source
    - Text """
    results = semantic_search(query)

    for r in results:
        print(f"Score: {r['score']:.3f}")
        print(f"Source: {r['source']}")
        print(r["text"])
        print("-" * 80)

Här har jag ett antal tester jag gjorde för att kontrollera min embedding och chunking. Detta för att utvärdera min chunking och justera n och overlap till en början och sedan se hur det såg ut med block-chunkingen.

In [26]:
query = "Hur startar jag maskinen?"

#retrieval_evaluation(query)

In [27]:
query = "Hur väljer jag program?"

#retrieval_evaluation(query)

In [28]:
query = "Hur rengör jag tvättmedelsfacket?"

#retrieval_evaluation(query)

In [29]:
query = "Hur ofta ska jag rengöra maskinen?"

#retrieval_evaluation(query)

In [30]:
query = "Vad ska jag tänka på innan jag startar maskinen?"

#retrieval_evaluation(query)

In [31]:
query = "Kan jag lämna luckan stängd efter tvätt?"

#retrieval_evaluation(query)

In [32]:
#query = "Hur undviker jag dålig lukt?"

#retrieval_evaluation(query)

Efter första utvärderingen (n=10 och overlap=2) så ser jag att embeddingen verkar fungera bra men chunksen är för stora då det kommer med ganska mycket orelevant information också. 
Så jag väljer att ändra till n=5 men behåller overlap=2.

Efter andra utvärderingen (n=5 och overlap=2) så blir retrivaln mycket sämre. Frågorna om vattnet tappade hittar mest orelevanta saker. Detta kan vara för att problemet och lösningen hamnar i olika chunks. Så mitt nästa test blir att öka overlap till 3 för att se vad det ger för resultat. 

Efter tredje utvärderingen (n=5 och overlap=3) så ser jag att det blir bättre men fortfarande inte bra, det fattas forfarande lite info, särskilt i de sam handlar om vatten. Sen ser jag också att vi får in massa av punkter i några av chunksen vilket jag inser är innehållsförteckningen. Så jag väljer att modifiera inläsningen så att jag inte läser in framsidan och de första orelevanta sidorna. Och mitt nästa test blir n=7 men behåller overlap=3.

Efter fjärde utvärderingen (n=7 och overlap=3) så ser det hyftast ut men jag inser också att jag ställer fel frågor för retrivlen då jag fokuserar förmycket på felsökning (en efterdyning från min första idé som var att göra en chatbot till servicetekniker ute i fältet) och manualen jag använder har ingen större felsökningsdel utan den handlar mer om hur man använder sig utav maskinen. Så efter att jag ändrade alla frågor till mer relevanta frågor så fick jag mycket bättre resultat och jag är nu redo att gå vidare till att göra själva chatboten.

Här tyckte jag att det var good enough för att se hur en LLM skulle kunna hantera det. Detta var ju innan jag förstod bättre ^^

# Generera svar

Här gör jag systemprompt för chatbotten så att den ska svara. Denna hade kunnat vara mer utsmyckad med mer instruktioner till den men jag tycker att den gör ett bra jobb ändå så kollade inte vidare på det.

In [33]:
system_prompt = """Du är en hjälpsam assistent.

Svara endast med hjälp av kontexten du får.
Om svaret inte finns i kontexten, säg exakt: "Det vet jag inte."
Gissa inte.

Användaren kan skriva "maskin", vilket alltid betyder "tvättmaskin".

Skriv tydligt, enkelt och dela upp svaret i stycken."""

Jag märkte i mina tester att när jag skriver en fråga till chatbotten så refererar jag tvättmaskinen som maskinen och det ville jag att den skulle förstå därför gjorde jag denna normaliseraren och sedan wrappen för att byta ut orden maskin, maskinen eller maskiner till tvättmaskin istället och då fungerar det att bara skriva maskin och det blir inte fel när den embeddar frågan. Jag la även till att användaren kan använda ordet maskin i system prompten men det är bara för att chatbotten ska kunna använda samma språk som användaren.

In [45]:
def normalize_query(query):
    """Byter orden maskin/en/er mot tvättmaskinen"""
    return re.sub(r"\bmaskin(en|er|)\b", "tvättmaskin", query.lower())

def semantic_search_wrapper(query):
    """Wrapper för semantik search med normalized query"""
    query = normalize_query(query)
    return semantic_search(query)

In [ ]:
def generate_user_prompt(query):
    """Skapar user prompt från användar inputen för att använda till LLMn"""
    results = semantic_search_wrapper(query)
    context = "\n\n".join([item["text"] for item in results])

    user_prompt = f"Fråga: {query}\n\n Kontext:\n{context}. Svara endast utifrån kontexten"
    return user_prompt

In [ ]:
def generate_response(system_prompt, user_message, model="gemini-2.5-flash"):
    """Skickar frågan till LLM-modellen för och retunerar dess svar"""
    response = client.models.generate_content(
        model=model,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt),
            contents=generate_user_prompt(user_message)
        )
    return response

Här under testar jag genereringen med färdiga frågor för att se vad för svar jag får från modellen.

Jag kommenterade bort de dessa kontrollfrågorna efter en del itereringar då jag har en bättre kontroll som inkluderar chunksen som jag började använda istället.

In [37]:
#user_message = "Hur får jag bort färgfläckar på kläderna?"

#print(generate_response(system_prompt, user_message).text)

In [38]:
# user_message = "Vad består gräs utav?"

# print(generate_response(system_prompt, user_message).text)

In [39]:
# user_message = "Min maskin låter konstigt, vad ska jag göra?"

# print(generate_response(system_prompt, user_message).text)

## Evaluering

Här så ville jag köra en riktigt evaluering av chatbotten så jag skapade valideringsdata med frågor och de svar jag tyckte chatbotten skulle svara för varje fråga. Jag la till en trick-question också för att kontrollera att den verkligen svarade "jag vet inte" när jag frågar något utanför kontexten.

In [ ]:
validation_data = [
    {
        "question": "Hur gör jag en fördröjd start?",
        "ideal_answer": "Genom att trycka på knappen så fördröjer du programstarten en timme per knapptryckning, detta kan du göra mellan 1 till 19 timmar. Tryck sedan på Start/Paus för att aktivera startfördröjningen" 
    },
    {
        "question": "Varför startar inte programmet?",
        "ideal_answer": "Detta kan bero på ett av följande: - Programmet valdes inte fullständigt. - Ströförsörjningen till tvättmaskinen saknas. - Vattenkranen är stängd. - Tillflödesslangen är knäckt. - Locket är inte riktigt stängt. - Filtret i vatteninloppet är igentäppta."
    },
    {
        "question": "Var hittar jag maskinens E-nr?",
        "ideal_answer": "Det hittar du bakom tvättmedelsfacket."
    },
    {
        "question": "Hur hög är Big Ben?",
        "ideal_answer": "Det vet jag inte."
    },    
]

evaluation_system_prompt = """Du är ett intelligent utvärderingssystem vars uppgift är att utvärdera en AI-assistents svar. 
    Om svaret är väldigt nära det önskade svaret, sätt poäng 1. 
    Om svaret är felaktigt eller inte bra nog, sätt poäng 0.
    Om svaret är delvis i linje med det önskade svaret, sätt poängen 0,5. 
    Motivera kort varför du sätter den poäng du gör"""

Jag skapade en for loop för att använda mig utav en LLM modell som skulle utvärdera min chatbot hur bra den var på att svara mot mina ideal svar. Men detta gjorde att min quota tog slut väldigt snabbt så jag valde att kommentera bort det och göra en manuell validering istället.

In [41]:
# for item in validation_data:
#    query = item['question']
#    ideal_answer = item['ideal_answer']

#    response = generate_response(system_prompt, query)

#    evaluation_prompt = f"""Fråga: {query}
#        AI-assistentens svar: {response.text}
#        Önskat svar: {ideal_answer}"""
    
#    evaluation_response = generate_response(evaluation_system_prompt, evaluation_prompt)

#    print((evaluation_response.text))

#    time.sleep(1)

Manuella utvärderingen:

In [42]:
for item in validation_data:
    query = item["question"]
    results = semantic_search(normalize_query(query))
    response = generate_response(system_prompt, query)

    print("FRÅGA:", query)
    print("\nRETRIEVED CHUNKS:")
    for r in results:
        print("-", r["text"][:500])
        print()

    print("MODELLSVAR:", response.text)
    print("IDEALSVAR:", item["ideal_answer"])
    print("=" * 80)

FRÅGA: Hur gör jag en fördröjd start?

RETRIEVED CHUNKS:
- Fördröjd start

Lägga in tvätt i efterhand + Ändra program

- Vrid programvredet

till ”Från”.
Öppna locket och ta ut tvätten.
Ställa in program
: eller på knappen
om du vill fördröja starten med mellan
1 och 19 timmar.
Display för programtid och
startfördröjning
Varvtalsväljare
Programtid, t.ex.

- Lockets öppningssäkring:

fördröjd start så är luckan spärrad under
hela startfördröjningstiden.
Om du vill öppna luckan under pågående
program, tryck kortvarigt på ”Start/Paus”
och vänta en till två minuter tills luckspärren
öppnas.
För att undvika brännskador är spärren
aktiv tills tvättvattnet har svalnat något.
Vattensäkring:
Under driften förebyggs eventuell
översvämning av den ständiga
vattennivåkontrollen.

- Ändra program

Om du har valt fel program av misstag:
• Ställ programvredet på ”Från”.
• Ställ programvredet på det nya programmet du
vill välja.
• Tryck på ”Start/Paus” för att starta.
Centrifugeringsvarvtalet kan ändra

Efter första utvärderingen av chatbotten så upptäckte jag att chunksen den hittade var väldigt dåliga. Många bestod av bara punkter så jag hoppade tillbaka till städningen av pdfen igen då jag verkar ha städat den lite för hårt förut. Sedan började jag också fundera på en annan chunking-metod än meningsbaserad som jag började med. Det var här jag insåg att jag inte kunde ha denna chunkingen och började undersöka andra typer av chunking och hittade då block chunking. Vilket bygger på att man gör block och sedan sätter ihop lagom många block i en chunk då flera block blir väldigt små. Så har jag fått iterera igenom det hela ett antal gånger tills jag vart nöjd och modellen svarade i stort sett precis som mina ideala svar. Då vart jag nöjd och gick vidare till den verkliga chatbotten.


# Chatboten

Här gör jag själva chatbotten, jag använder mig utav en enkel input att ta in svaret från användaren. 
Jag valde att inte ta den vidare till en app för att jag kände att jag behövde tid till annat. Men kanske nu i efterhand så kommer jag göra en app för att ha i min webb-portfolio och visa upp vad jag skapat :)

In [44]:
if __name__ == "__main__":
    print("*** Tvätthjälpen din chatbot för din Bosch tvättmaskin***")
    print("Skriv <q> för att avsluta chatten")
    print('-'*80)
    while True:
        prompt = input("Användare: ")
        if prompt == 'q':
          break
        else:
            response = generate_response(system_prompt, prompt)
            print(f'Användare: {prompt} \nTvätthjälpen: {response.text}')
            print('-'*80)

*** Tvätthjälpen din chatbot för din Bosch tvättmaskin***
Skriv <q> för att avsluta chatten
--------------------------------------------------------------------------------


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

Detta har varit ett väldigt lärorikt projekt och åter igen fick jag vittna om skit in skit ut och hur viktigt det är att ta hand om datan innan man börjar försöka göra coola saker med den ^^

Den här modellen hade kunnat användas utav tvättmaskinsföretag som en extra support åt sina kunder. 
Nu har jag enbart använt mig utav en manual men företaget skulle kunna ha alla sina maskiner inlagda för att ge support över hela deras produktportfölj. Något man också skulle kunna göra är att skriva mer riktlinjer för själva språkmodellen för att kunna bli ännu mer inriktad kundsupport. 

Detta tror jag hade lett till ett högre förtroende för företaget, minska trycket på kundsupporten så de kan fokusera på de mer komplexa frågorna men också så tror jag att man skulle kunna få in merförsäljning också. Mer försäljningen skulle kunna vara att man gör detta till en app och i appen kan man beställa reservdelar man själv kan byta eller om problemet verkar för komplext antingen direkt boka en servicetekniker eller om man först vill koppla till en riktig kundsupport som utvärderar problemet och bokar service om det behövs.